#### APPLIED — SGD vs Adam 비교

In [2]:
import keras
import tensorflow as tf

In [3]:
(train_input, train_target), (test_input, test_target) = \
    keras.datasets.fashion_mnist.load_data()
    
train_scaled = train_input / 255.0

In [4]:
def make_model():
    m = keras.Sequential()
    m.add(keras.layers.Input(shape=(28, 28)))
    m.add(keras.layers.Flatten())
    m.add(keras.layers.Dense(100, activation='relu'))
    m.add(keras.layers.Dense(10, activation='softmax'))
    return m

for opt_name, opt in {
    'SGD (기본)':   keras.optimizers.SGD(),
    'SGD (lr=0.1)': keras.optimizers.SGD(learning_rate=0.1),
    'Adam':         keras.optimizers.Adam(),
}.items():
    keras.utils.set_random_seed(42)
    tf.config.experimental.enable_op_determinism()
    model = make_model()
    model.compile(optimizer=opt, 
                  loss='sparse_categorical_crossentropy', 
                  metrics=['accuracy'])
    model.fit(train_scaled, train_target, epochs=5, verbose=0)
    _, val_acc = model.evaluate(train_scaled, train_target, verbose=0)
    print(f"{opt_name}: {val_acc:.4f}")

SGD (기본): 0.8494
SGD (lr=0.1): 0.8823
Adam: 0.8941


#### CHALLENGE

**Part 1: Adam lr 조정**

In [ ]:
for lr in [0.1, 0.01, 0.001, 0.0001]:
    # 결과 비교
    # lr=0.1 → 발산? lr=0.0001 → 느림?
    model = keras.Sequential([
        keras.layers.Input(shape=(28, 28)),
        keras.layers.Flatten(),
        keras.layers.Dense(100, activation='relu'),
        keras.layers.Dense(10, activation='softmax')
    ])

    model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr), 
                loss='sparse_categorical_crossentropy', 
                metrics=['accuracy'])
    history = model.fit(train_scaled, train_target, 
                        epochs=5,
                        validation_split=0.2)
    
    print(f"Final val_accuracy: {history.history['val_accuracy'][-1]:.4f}")

Epoch 1/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.4106 - loss: 1.7175 - val_accuracy: 0.4370 - val_loss: 1.3946
Epoch 2/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.3341 - loss: 1.7269 - val_accuracy: 0.3434 - val_loss: 1.6456
Epoch 3/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.3060 - loss: 1.7863 - val_accuracy: 0.2883 - val_loss: 1.8386
Epoch 4/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.3113 - loss: 1.7736 - val_accuracy: 0.3215 - val_loss: 1.7046
Epoch 5/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.2039 - loss: 2.0466 - val_accuracy: 0.1880 - val_loss: 2.1001
Final val_accuracy: 0.1880
Epoch 1/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.8040 - loss: 0.5424 - val_accuracy: 0.8346 - val_loss: 0.4535
Epoch 2/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.8387 - loss: 0.4461 - val_accuracy: 0.8375 - val_loss: 0.4439
Epoch 3/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.849

**Part 2: CIFAR-10 Dense 한계**

In [7]:
(train_input, train_target), (test_input, test_target) = \
    keras.datasets.cifar10.load_data()

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 13s 0us/step


In [11]:
train_input.shape, train_target.shape, test_input.shape, test_target.shape

((50000, 32, 32, 3), (50000, 1), (10000, 32, 32, 3), (10000, 1))

In [13]:
train_scaled = train_input / 255.0

In [15]:
model = keras.Sequential([
        keras.layers.Input(shape=(32, 32, 3)),
        keras.layers.Flatten(),
        keras.layers.Dense(100, activation='relu'),
        keras.layers.Dense(10, activation='softmax')
    ])

model.compile(optimizer='adam', 
            loss='sparse_categorical_crossentropy', 
            metrics=['accuracy'])
model.fit(train_scaled, train_target, 
          epochs=5,
          validation_split=0.2)

Epoch 1/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.3081 - loss: 1.9278 - val_accuracy: 0.3330 - val_loss: 1.8836
Epoch 2/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.3540 - loss: 1.8094 - val_accuracy: 0.3440 - val_loss: 1.8504
Epoch 3/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.3667 - loss: 1.7737 - val_accuracy: 0.3527 - val_loss: 1.8226
Epoch 4/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.3754 - loss: 1.7534 - val_accuracy: 0.3593 - val_loss: 1.8053
Epoch 5/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.3793 - loss: 1.7391 - val_accuracy: 0.3693 - val_loss: 1.7845
